In [ ]:
# =============================================================================
# Final Test Set Evaluation — 2023–2024
# Models: model_ignition_final.pkl
#         model_xgb_cause_binary_human.pkl
#         model_xgb_cause_binary_natural.pkl
# Test data: ml_data_2023-2024 folder
# THIS IS THE LOCKED TEST SET — run only once after all modelling decisions
# =============================================================================

import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
import glob
import os
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score,
                             classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             average_precision_score)
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
TEST_DATA_DIR  = r'C:\Users\lambe\RU_Thesis_2026\ml_data_2023-2024'
MODELS_DIR     = 'models'

DYNAMIC_FEATURES = [
    'temp_max', 'rh_min', 'vpd_mean', 'wind_speed_mean',
    'precip_sum', 'solar_rad_mj', 'rh_7d', 'temp_7d',
    'precip_30d', 'recovery_value'
]
STATIC_FEATURES = [
    'elevation_m', 'slope_deg', 'land_cover',
    'pop_density', 'road_proximity_m'
]
FEATURES       = DYNAMIC_FEATURES + STATIC_FEATURES
TEST_YEARS     = [2023, 2024]
DOY_START_HARD = 91
DOY_END_HARD   = 310
RANDOM_STATE   = 42

CAUSE_NAMES = {0: 'Uncertain', 2: 'Human', 3: 'Natural'}
CAUSE_TRUE  = {
    1: 'Undetermined', 2: 'Human',
    3: 'Natural', 4: 'H-PB', 5: 'Re-ignition'
}

# =============================================================================
# STEP 1: Load models
# =============================================================================
print("="*65)
print("FINAL TEST SET EVALUATION — 2023–2024")
print("THIS IS THE LOCKED TEST SET")
print("="*65)

print("\nSTEP 1: Loading models...")

bundle_ign     = joblib.load(
    os.path.join(MODELS_DIR, 'model_ignition_final.pkl'))
model_ign      = bundle_ign['model']
CV_THRESHOLD   = bundle_ign['cv_threshold']

bundle_human   = joblib.load(
    os.path.join(MODELS_DIR, 'model_xgb_cause_binary_human.pkl'))
model_human    = bundle_human['model']
THR_HUMAN      = bundle_human.get('cv_threshold', 0.5)

bundle_natural = joblib.load(
    os.path.join(MODELS_DIR, 'model_xgb_cause_binary_natural.pkl'))
model_natural  = bundle_natural['model']
THR_NATURAL    = bundle_natural.get('cv_threshold', 0.5)

print(f"  Ignition model    : model_ignition_final.pkl")
print(f"  CV threshold      : {CV_THRESHOLD:.4f}")
print(f"  Human model       : model_xgb_cause_binary_human.pkl")
print(f"  Human CV thr      : {THR_HUMAN:.4f}")
print(f"  Natural model     : model_xgb_cause_binary_natural.pkl")
print(f"  Natural CV thr    : {THR_NATURAL:.4f}")

# =============================================================================
# STEP 2: Load test data
# =============================================================================
print("\n" + "="*65)
print("STEP 2: Loading test data (2023–2024) — LOCKED TEST SET")
print("="*65)

KEEP_COLS = ['date'] + FEATURES + ['target_ignition', 'fire_cause']

# ── Detect file naming pattern ─────────────────────────────────────────────────
# Try both ML_test_YYYYMM.csv and ML_train_YYYYMM.csv patterns
sample_files_test  = sorted(glob.glob(
    os.path.join(TEST_DATA_DIR, 'ML_test_*.csv')))
sample_files_train = sorted(glob.glob(
    os.path.join(TEST_DATA_DIR, 'ML_train_*.csv')))
sample_files_any   = sorted(glob.glob(
    os.path.join(TEST_DATA_DIR, '*.csv')))

print(f"\n  Files found:")
print(f"    ML_test_*.csv  : {len(sample_files_test)}")
print(f"    ML_train_*.csv : {len(sample_files_train)}")
print(f"    Any CSV        : {len(sample_files_any)}")

if sample_files_any:
    print(f"  First file: {os.path.basename(sample_files_any[0])}")
    # Peek at columns
    peek = pd.read_csv(sample_files_any[0], nrows=2)
    print(f"  Columns: {peek.columns.tolist()}")

# Choose the correct pattern
if sample_files_test:
    file_pattern = 'ML_test'
elif sample_files_train:
    file_pattern = 'ML_train'
else:
    # Use whatever pattern exists
    basename     = os.path.basename(sample_files_any[0])
    file_pattern = '_'.join(basename.split('_')[:2])

print(f"\n  Using file pattern: {file_pattern}_YYYYMM.csv")

test_dfs = []
for year in TEST_YEARS:
    year_files = sorted(glob.glob(
        os.path.join(TEST_DATA_DIR,
                     f'{file_pattern}_{year}??.csv')))

    if not year_files:
        # Fallback: any file containing the year
        year_files = sorted(glob.glob(
            os.path.join(TEST_DATA_DIR, f'*{year}*.csv')))

    print(f"\n  {year}: {len(year_files)} files found")

    for f in year_files:
        try:
            # Load only columns that exist
            df_peek  = pd.read_csv(f, nrows=0)
            existing = [c for c in KEEP_COLS if c in df_peek.columns]
            missing  = [c for c in KEEP_COLS if c not in df_peek.columns]
            if missing:
                print(f"    WARNING: missing columns in "
                      f"{os.path.basename(f)}: {missing}")

            df = pd.read_csv(f, usecols=existing, on_bad_lines='skip')
            test_dfs.append(df)
            n_fire = (df['target_ignition'] == 1).sum() \
                     if 'target_ignition' in df.columns else 0
            print(f"    {os.path.basename(f)}: "
                  f"{len(df):,} rows  ({n_fire:,} fires)")
        except Exception as e:
            print(f"    WARNING: {os.path.basename(f)}: {e}")

if not test_dfs:
    raise FileNotFoundError(
        f"No test CSV files found in {TEST_DATA_DIR}\n"
        f"Check that the folder exists and contains CSV files.")

df_test = pd.concat(test_dfs, ignore_index=True)

# ── Clean and filter ───────────────────────────────────────────────────────────
df_test['date'] = df_test['date'].astype(int).astype(str)
df_test['doy']  = pd.to_datetime(
    df_test['date'], format='%Y%m%d', errors='coerce').dt.dayofyear
df_test = df_test[(df_test['doy'] >= DOY_START_HARD) &
                  (df_test['doy'] <= DOY_END_HARD)].drop(columns='doy')
df_test = df_test.dropna(
    subset=FEATURES + ['target_ignition', 'fire_cause'])
df_test['target_ignition'] = df_test['target_ignition'].astype(int)
df_test['fire_cause']      = df_test['fire_cause'].astype(int)
df_test = df_test.sort_values('date').reset_index(drop=True)

n_fire   = (df_test['target_ignition'] == 1).sum()
n_nofire = (df_test['target_ignition'] == 0).sum()
ratio    = n_nofire / max(n_fire, 1)

print(f"\n  Test set summary:")
print(f"    Total rows    : {len(df_test):,}")
print(f"    Fire pixels   : {n_fire:,}")
print(f"    No-fire pixels: {n_nofire:,}")
print(f"    Class ratio   : {ratio:.0f}:1")
print(f"    Date range    : "
      f"{df_test['date'].min()} – {df_test['date'].max()}")
print(f"    Unique dates  : {df_test['date'].nunique():,}")

# Cause distribution in test set
print(f"\n  Cause distribution in test fire pixels:")
fire_test = df_test[df_test['target_ignition'] == 1]
for code, label in CAUSE_TRUE.items():
    n = (fire_test['fire_cause'] == code).sum()
    if n > 0:
        print(f"    Code {code} ({label:<15}): {n:,}  "
              f"({n/len(fire_test)*100:.1f}%)")

# =============================================================================
# STEP 3: Ignition model evaluation on test set
# =============================================================================
print("\n" + "="*65)
print("STEP 3: Ignition model — test set evaluation")
print(f"  CV threshold: {CV_THRESHOLD:.4f}")
print("="*65)

X_test      = df_test[FEATURES].values
y_true_test = df_test['target_ignition'].values

y_prob_test = model_ign.predict_proba(X_test)[:, 1]
y_pred_test = (y_prob_test >= CV_THRESHOLD).astype(int)

tn, fp, fn, tp = confusion_matrix(y_true_test, y_pred_test).ravel()
auc_ign  = roc_auc_score(y_true_test, y_prob_test)
ap_ign   = average_precision_score(y_true_test, y_prob_test)
bal_ign  = balanced_accuracy_score(y_true_test, y_pred_test)
rec_ign  = tp / (tp + fn)
prec_ign = tp / (tp + fp) if (tp + fp) > 0 else 0
fpr_ign  = fp / (fp + tn)
f1_ign   = f1_score(y_true_test, y_pred_test, zero_division=0)

print(f"\n  IGNITION MODEL — TEST SET RESULTS (2023–2024)")
print(f"  {'─'*50}")
print(f"  ROC-AUC           : {auc_ign:.4f}")
print(f"  Avg Precision (AP): {ap_ign:.4f}")
print(f"  Balanced accuracy : {bal_ign:.4f}")
print(f"  Recall (fire)     : {rec_ign:.4f}  "
      f"({tp} of {tp+fn} fires detected)")
print(f"  Precision (fire)  : {prec_ign:.4f}")
print(f"  F1 score          : {f1_ign:.4f}")
print(f"  FPR               : {fpr_ign:.4f}  "
      f"({fpr_ign*100:.1f}% of no-fire days flagged)")
print(f"  TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")
print(f"\n  Classification report:")
print(classification_report(y_true_test, y_pred_test,
                            target_names=['No fire', 'Fire'],
                            digits=4, zero_division=0))

# ── Compare test vs validation performance ─────────────────────────────────────
print(f"\n  Comparison: Validation (2021–2022) vs Test (2023–2024)")
print(f"  {'Metric':<22} {'Val':>10} {'Test':>10} {'Δ':>8}")
print(f"  {'─'*52}")
# Validation metrics from bundle if stored, otherwise from known values
val_metrics = {
    'ROC-AUC'  : bundle_ign.get('val_auc',  0.929),
    'Recall'   : 0.773,
    'Precision': 0.002,
    'FPR'      : 0.099,
}
test_metrics = {
    'ROC-AUC'  : auc_ign,
    'Recall'   : rec_ign,
    'Precision': prec_ign,
    'FPR'      : fpr_ign,
}
for metric in val_metrics:
    v = val_metrics[metric]
    t = test_metrics[metric]
    print(f"  {metric:<22} {v:>10.4f} {t:>10.4f} "
          f"{t-v:>+8.4f}")

# =============================================================================
# STEP 4: Cause model evaluation on test set
# =============================================================================
print("\n" + "="*65)
print("STEP 4: Cause model — test set evaluation")
print("="*65)

# Predicted fire pixels in test set
mask_pred_test    = y_pred_test == 1
X_pred_fire_test  = X_test[mask_pred_test]
y_cause_pred_true = df_test.loc[mask_pred_test, 'fire_cause'].values
y_ign_pred_true   = df_test.loc[mask_pred_test, 'target_ignition'].values

# Actual fires within predicted set
actual_mask_test   = y_ign_pred_true == 1
X_cause_test_actual = X_pred_fire_test[actual_mask_test]
y_cause_test_actual = y_cause_pred_true[actual_mask_test]

print(f"\n  Predicted fire pixels    : {mask_pred_test.sum():,}")
print(f"  Actual fires (TP)        : {actual_mask_test.sum():,}")
print(f"  False positives          : "
      f"{(~actual_mask_test).sum():,}")

# ── Binary model evaluation ────────────────────────────────────────────────────
binary_eval_test = {}

for model_key, model, pos_label, pos_name, neg_name, thr in [
    ('human',   model_human,   2, 'Human',
     'Other (Natural/Undetermined)',
     THR_HUMAN),
    ('natural', model_natural, 3, 'Natural (lightning)',
     'Other (Human/Undetermined)',
     THR_NATURAL),
]:
    y_bin_true = (y_cause_test_actual == pos_label).astype(int)
    y_bin_prob = model.predict_proba(X_cause_test_actual)[:, 1]
    y_bin_pred = (y_bin_prob >= thr).astype(int)

    if y_bin_true.sum() == 0:
        print(f"\n  WARNING: no {pos_name} fires in test set — "
              f"skipping {model_key} evaluation")
        continue

    tn_c, fp_c, fn_c, tp_c = confusion_matrix(
        y_bin_true, y_bin_pred).ravel()
    auc_c  = roc_auc_score(y_bin_true, y_bin_prob)
    ap_c   = average_precision_score(y_bin_true, y_bin_prob)
    bal_c  = balanced_accuracy_score(y_bin_true, y_bin_pred)
    prec_c = tp_c / (tp_c + fp_c) if (tp_c + fp_c) > 0 else 0
    rec_c  = tp_c / (tp_c + fn_c) if (tp_c + fn_c) > 0 else 0
    f1_c   = (2 * prec_c * rec_c / (prec_c + rec_c)
              if (prec_c + rec_c) > 0 else 0)
    fpr_c  = fp_c / (fp_c + tn_c) if (fp_c + tn_c) > 0 else 0
    spec_c = tn_c / (tn_c + fp_c) if (tn_c + fp_c) > 0 else 0

    binary_eval_test[model_key] = {
        'pos_name': pos_name,
        'bal_acc' : bal_c,
        'val_auc' : auc_c,
        'ap'      : ap_c,
        'prec'    : prec_c,
        'rec'     : rec_c,
        'spec'    : spec_c,
        'f1'      : f1_c,
        'fpr'     : fpr_c,
        'tp': tp_c, 'fp': fp_c, 'fn': fn_c, 'tn': tn_c,
        'n_pos'   : int(y_bin_true.sum()),
        'n_total' : len(y_bin_true),
    }

    print(f"\n  {'─'*55}")
    print(f"  {pos_name} vs Other  (threshold = {thr:.4f})")
    print(f"  Evaluated on actual fires in predicted set "
          f"(n={len(y_bin_true):,})")
    print(f"  Positive class: {n:,} fires  "
          f"({y_bin_true.mean()*100:.1f}%)")
    print(f"  {'─'*55}")
    print(f"\n  Confusion matrix:")
    print(f"  {'':25} {'Pred Other':>14} {'Pred '+pos_name[:7]:>14}")
    print(f"  {'─'*55}")
    print(f"  {'Actual Other':<25} {tn_c:>14,} {fp_c:>14,}")
    print(f"  {'Actual '+pos_name:<25} {fn_c:>14,} {tp_c:>14,}")
    print(f"  {'─'*55}")
    print(f"\n  Performance metrics:")
    print(f"    ROC-AUC           : {auc_c:.4f}")
    print(f"    Avg Precision (AP): {ap_c:.4f}")
    print(f"    Balanced accuracy : {bal_c:.4f}")
    print(f"    Precision         : {prec_c:.4f}")
    print(f"    Recall            : {rec_c:.4f}")
    print(f"    Specificity       : {spec_c:.4f}")
    print(f"    F1 score          : {f1_c:.4f}")
    print(f"    FPR               : {fpr_c:.4f}")
    print(f"    TP={tp_c}  FP={fp_c}  FN={fn_c}  TN={tn_c}")
    print(classification_report(
        y_bin_true, y_bin_pred,
        target_names=[neg_name, pos_name],
        digits=4, zero_division=0))

# ── Side-by-side binary comparison ────────────────────────────────────────────
if len(binary_eval_test) == 2:
    print(f"\n{'='*65}")
    print("BINARY CAUSE MODEL COMPARISON — TEST SET")
    print(f"{'='*65}")
    print(f"  {'Metric':<22} {'Human vs Other':>16} "
          f"{'Natural vs Other':>18}")
    print(f"  {'─'*58}")
    for label, key in [
        ('Balanced accuracy', 'bal_acc'),
        ('ROC-AUC',           'val_auc'),
        ('Avg Precision',     'ap'),
        ('Precision',         'prec'),
        ('Recall',            'rec'),
        ('Specificity',       'spec'),
        ('F1 score',          'f1'),
        ('FPR',               'fpr'),
    ]:
        h = binary_eval_test.get('human',   {}).get(key, float('nan'))
        n = binary_eval_test.get('natural', {}).get(key, float('nan'))
        print(f"  {label:<22} {h:>16.4f} {n:>18.4f}")
    print(f"  {'─'*58}")
    for label, key in [
        ('TP','tp'), ('FP','fp'), ('FN','fn'), ('TN','tn')]:
        h = binary_eval_test.get('human',   {}).get(key, 0)
        n = binary_eval_test.get('natural', {}).get(key, 0)
        print(f"  {label:<22} {h:>16,} {n:>18,}")

# =============================================================================
# STEP 5: Three-way prediction on test set
# =============================================================================
print(f"\n{'='*65}")
print("STEP 5: Three-way cause prediction — test set")
print(f"{'='*65}")

p_hum_test = model_human.predict_proba(X_pred_fire_test)[:, 1]
p_nat_test = model_natural.predict_proba(X_pred_fire_test)[:, 1]

def predict_three_way(p_hum, p_nat, thr_hum, thr_nat):
    pred = np.zeros(len(p_hum), dtype=int)
    pred[(p_nat > p_hum)  & (p_nat >= thr_nat)] = 3
    pred[(p_hum >= p_nat) & (p_hum >= thr_hum)] = 2
    return pred

y_cause_pred_test = predict_three_way(
    p_hum_test, p_nat_test, THR_HUMAN, THR_NATURAL)

# Evaluate on actual fires
y_true_eval_test = y_cause_pred_true[actual_mask_test]
y_pred_eval_test = y_cause_pred_test[actual_mask_test]

n_total_test  = len(y_true_eval_test)
n_uncertain_t = (y_pred_eval_test == 0).sum()
n_known_t     = (y_pred_eval_test != 0).sum()
correct_t     = (y_pred_eval_test == y_true_eval_test).sum()
correct_known = ((y_pred_eval_test == y_true_eval_test) &
                 (y_pred_eval_test != 0)).sum()

overall_acc_t = correct_t / n_total_test if n_total_test > 0 else 0
known_acc_t   = correct_known / n_known_t if n_known_t > 0 else 0

hn_mask_t    = np.isin(y_true_eval_test, [2, 3])
bal_acc_3way_t = balanced_accuracy_score(
    y_true_eval_test[hn_mask_t],
    y_pred_eval_test[hn_mask_t]) if hn_mask_t.sum() > 0 else 0

print(f"\n  Prediction distribution (all predicted fire pixels):")
for code, name in [(2,'Human'), (3,'Natural'), (0,'Uncertain')]:
    n = (y_cause_pred_test == code).sum()
    print(f"    {name:<12}: {n:,}  "
          f"({n/len(y_cause_pred_test)*100:.1f}%)")

target_names_map = {0: 'Uncertain', 2: 'Human', 3: 'Natural'}
present_classes  = sorted(set(
    list(y_true_eval_test[np.isin(y_true_eval_test, [2, 3])]) +
    list(y_pred_eval_test)))
present_classes  = [c for c in present_classes
                    if c in target_names_map]

print(f"\n  Classification report (actual fires in predicted set):")
print(classification_report(
    y_true_eval_test, y_pred_eval_test,
    labels       = present_classes,
    target_names = [target_names_map[c] for c in present_classes],
    digits=4, zero_division=0))

print(f"  Overall accuracy (Uncertain = wrong):")
print(f"    {correct_t:,} / {n_total_test:,}  =  "
      f"{overall_acc_t:.4f}  ({overall_acc_t*100:.2f}%)")
print(f"\n  Classified-only accuracy (excl. Uncertain):")
print(f"    Classified : {n_known_t:,} / {n_total_test:,}  "
      f"({n_known_t/n_total_test*100:.1f}%)")
print(f"    Correct    : {correct_known:,} / {n_known_t:,}  "
      f"=  {known_acc_t:.4f}  ({known_acc_t*100:.2f}%)")
print(f"\n  Balanced accuracy (Human + Natural only):")
print(f"    {bal_acc_3way_t:.4f}")

print(f"\n  Per-class breakdown:")
print(f"  {'True class':<16} {'N actual':>10} {'Pred Human':>12} "
      f"{'Pred Natural':>14} {'Pred Uncertain':>16} {'Accuracy':>10}")
print(f"  {'─'*80}")
for true_code, true_name in [(2,'Human'), (3,'Natural')]:
    mask_true  = y_true_eval_test == true_code
    n_true     = mask_true.sum()
    if n_true == 0:
        continue
    n_hum = ((y_pred_eval_test == 2) & mask_true).sum()
    n_nat = ((y_pred_eval_test == 3) & mask_true).sum()
    n_unc = ((y_pred_eval_test == 0) & mask_true).sum()
    acc_c = ((y_pred_eval_test == true_code) &
              mask_true).sum() / n_true
    print(f"  {true_name:<16} {n_true:>10,} {n_hum:>12,} "
          f"{n_nat:>14,} {n_unc:>16,} {acc_c:>10.4f}")
print(f"  {'─'*80}")

# =============================================================================
# STEP 6: Final summary table
# =============================================================================
print(f"\n{'='*65}")
print("FINAL TEST SET EVALUATION SUMMARY")
print("Models trained: 2010–2020  |  Validated: 2021–2022")
print("Tested: 2023–2024  (LOCKED — first and only evaluation)")
print(f"{'='*65}")

print(f"\n  IGNITION MODEL (XGBoost)")
print(f"  {'─'*50}")
print(f"    Test observations : {len(df_test):,}")
print(f"    Fire events       : {n_fire:,}")
print(f"    CV threshold      : {CV_THRESHOLD:.4f}")
print(f"    ROC-AUC           : {auc_ign:.4f}")
print(f"    Balanced accuracy : {bal_ign:.4f}")
print(f"    Recall            : {rec_ign:.4f}  ({tp}/{tp+fn} fires)")
print(f"    Precision         : {prec_ign:.4f}")
print(f"    FPR               : {fpr_ign:.4f}")
print(f"    TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")

print(f"\n  CAUSE MODELS (XGBoost binary + three-way rule)")
print(f"  {'─'*50}")
if binary_eval_test:
    for key, e in binary_eval_test.items():
        print(f"    {e['pos_name']:<25}: "
              f"AUC={e['val_auc']:.4f}  "
              f"Bal.Acc={e['bal_acc']:.4f}  "
              f"F1={e['f1']:.4f}")
print(f"\n  THREE-WAY ACCURACY")
print(f"    Overall accuracy      : {overall_acc_t:.4f}")
print(f"    Classified-only acc   : {known_acc_t:.4f}")
print(f"    Balanced accuracy     : {bal_acc_3way_t:.4f}")
print(f"    Uncertain rate        : "
      f"{n_uncertain_t/n_total_test*100:.1f}%  "
      f"({n_uncertain_t:,}/{n_total_test:,})")

# ── LaTeX summary ──────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("LaTeX — Test Set Performance Summary")
print(f"{'='*65}")
print(f"""
\\begin{{table}}[h]
\\centering
\\caption{{Final test set evaluation (2023--2024) of the XGBoost
         ignition and binary cause models. The test set was held
         out throughout model development and evaluated once upon
         completion of all modelling decisions.}}
\\label{{tab:test_set_results}}
\\renewcommand{{\\arraystretch}}{{1.3}}
\\begin{{tabular}}{{>{{\\raggedright\\arraybackslash}}p{{4.5cm}} r r r}}
\\hline
\\textbf{{Metric}} &
\\textbf{{Ignition}} &
\\textbf{{Human vs Other}} &
\\textbf{{Natural vs Other}} \\\\
\\hline
\\multicolumn{{4}}{{l}}{{\\textit{{Threshold-independent}}}} \\\\
ROC-AUC
  & {auc_ign:.4f}
  & {binary_eval_test.get('human',{}).get('val_auc', float('nan')):.4f}
  & {binary_eval_test.get('natural',{}).get('val_auc', float('nan')):.4f} \\\\
Avg Precision (AP)
  & {ap_ign:.4f}
  & {binary_eval_test.get('human',{}).get('ap', float('nan')):.4f}
  & {binary_eval_test.get('natural',{}).get('ap', float('nan')):.4f} \\\\
\\hline
\\multicolumn{{4}}{{l}}{{\\textit{{At CV threshold}}}} \\\\
Balanced accuracy
  & {bal_ign:.4f}
  & {binary_eval_test.get('human',{}).get('bal_acc', float('nan')):.4f}
  & {binary_eval_test.get('natural',{}).get('bal_acc', float('nan')):.4f} \\\\
Recall
  & {rec_ign:.4f}
  & {binary_eval_test.get('human',{}).get('rec', float('nan')):.4f}
  & {binary_eval_test.get('natural',{}).get('rec', float('nan')):.4f} \\\\
Precision
  & {prec_ign:.4f}
  & {binary_eval_test.get('human',{}).get('prec', float('nan')):.4f}
  & {binary_eval_test.get('natural',{}).get('prec', float('nan')):.4f} \\\\
F1 score
  & {f1_ign:.4f}
  & {binary_eval_test.get('human',{}).get('f1', float('nan')):.4f}
  & {binary_eval_test.get('natural',{}).get('f1', float('nan')):.4f} \\\\
FPR
  & {fpr_ign:.4f}
  & {binary_eval_test.get('human',{}).get('fpr', float('nan')):.4f}
  & {binary_eval_test.get('natural',{}).get('fpr', float('nan')):.4f} \\\\
\\hline
\\multicolumn{{4}}{{l}}{{\\textit{{Three-way cause prediction}}}} \\\\
Overall accuracy $\\dagger$
  & \\multicolumn{{3}}{{c}}{{{overall_acc_t:.4f}}} \\\\
Classified-only acc $\\ddagger$
  & \\multicolumn{{3}}{{c}}{{{known_acc_t:.4f}}} \\\\
Balanced accuracy
  & \\multicolumn{{3}}{{c}}{{{bal_acc_3way_t:.4f}}} \\\\
Uncertain rate
  & \\multicolumn{{3}}{{c}}{{{n_uncertain_t/n_total_test*100:.1f}\\%}} \\\\
\\hline
\\end{{tabular}}
\\end{{table}}
""")

print("\nTest set evaluation complete.")
print("Results above represent the final, unbiased model performance.")